# Tadhkirat — Disease Treatment English Translation Pipeline

## What this does:
- Loads `disease_treatment_v1_fixed.csv`
- Translates disease_or_condition, treatment_method, expected_effect to English
- Keeps arabic_name, entry_id unchanged
- Handles placeholders correctly
- Saves after every 50 rows — safe to disconnect anytime
- Output: `disease_treatment_english_v1.csv`

## Cell 1 — Mount Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_FOLDER  = '/content/drive/MyDrive/tadhkirat_dawood'
INPUT_CSV     = f'{DRIVE_FOLDER}/disease_treatment_v1_fixed.csv'
OUTPUT_CSV    = f'{DRIVE_FOLDER}/disease_treatment_english_v1.csv'
PROGRESS_FILE = f'{DRIVE_FOLDER}/translation_progress_v1.txt'

PROJECT_ID = 'project-7f940e6f-ba1d-404b-948'
LOCATION   = 'global'
MODEL_ID   = 'gemini-3-flash-preview'

# How many rows to translate per Gemini call
# Batching saves API calls and is faster
BATCH_SIZE = 20

print(f'Input  : {"✓ Found" if os.path.exists(INPUT_CSV) else "✗ NOT FOUND"}')
print(f'Output : {OUTPUT_CSV}')

## Cell 2 — Install & Authenticate

In [ ]:
!pip install google-genai pandas openpyxl -q

from google.colab import auth
auth.authenticate_user()

from google import genai
from google.genai import types
import pandas as pd
import json
import time
import re

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

test = client.models.generate_content(
    model=MODEL_ID,
    contents='Say yes only'
)
print(f'✓ Gemini ready: {test.text.strip()}')

## Cell 3 — Translation Function (Batch)

In [ ]:
# Known placeholder translations — handle without API call
PLACEHOLDER_MAP = {
    'كما ذُكر في الكتاب':           'As mentioned in the book',
    'كما يُستعمل النبات عموماً':    'As the plant is generally used',
    'كما ذكر في الكتاب':            'As mentioned in the book',
    'كما يستعمل النبات عموماً':     'As the plant is generally used',
    'لا توجد أمراض محددة مذكورة':   'No specific diseases mentioned',
    'nan':                           '',
    '':                              ''
}

def translate_placeholder(text: str) -> str:
    """Returns translation if text is a known placeholder, else None."""
    if not text or str(text).strip() in ['', 'nan']:
        return ''
    text_stripped = str(text).strip()
    return PLACEHOLDER_MAP.get(text_stripped, None)


def translate_batch(rows: list) -> list:
    """
    Translates a batch of rows in one Gemini call.
    Each row is a dict with: disease, treatment, effect
    Returns list of dicts with English versions.
    """

    # Build batch input — number each item
    batch_text = ''
    for i, row in enumerate(rows):
        batch_text += f"""
ITEM_{i+1}:
DISEASE: {row['disease']}
TREATMENT: {row['treatment']}
EFFECT: {row['effect']}
"""

    prompt = f"""You are a medical translator specializing in classical Islamic medicine.
Translate the following Arabic medical terms to English.

RULES:
1. Translate disease names to their standard modern English medical equivalent
2. Translate treatment methods accurately and naturally
3. Translate effects/benefits accurately
4. Keep medical terminology precise
5. If a field is empty or says 'nan' return empty string
6. Do NOT translate herb names that appear in treatment descriptions
7. Keep numbers and measurements as is
8. Respond ONLY in the exact format shown — no extra text

{batch_text}

Respond in this EXACT format for each item:
ITEM_1:
DISEASE_EN: [English translation]
TREATMENT_EN: [English translation]
EFFECT_EN: [English translation]

ITEM_2:
DISEASE_EN: [English translation]
TREATMENT_EN: [English translation]
EFFECT_EN: [English translation]

Continue for all items."""

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt
        )
        text = response.text.strip()

        # Parse the response
        results = []
        # Split by ITEM_ markers
        items = re.split(r'ITEM_\d+:', text)
        items = [item.strip() for item in items if item.strip()]

        for i, item_text in enumerate(items):
            disease_en  = ''
            treatment_en = ''
            effect_en   = ''

            for line in item_text.split('\n'):
                line = line.strip()
                if line.startswith('DISEASE_EN:'):
                    disease_en = line.split(':', 1)[1].strip()
                elif line.startswith('TREATMENT_EN:'):
                    treatment_en = line.split(':', 1)[1].strip()
                elif line.startswith('EFFECT_EN:'):
                    effect_en = line.split(':', 1)[1].strip()

            results.append({
                'disease_en':   disease_en,
                'treatment_en': treatment_en,
                'effect_en':    effect_en
            })

        # If we got fewer results than input pad with empty
        while len(results) < len(rows):
            results.append({
                'disease_en':   '',
                'treatment_en': '',
                'effect_en':    ''
            })

        return results

    except Exception as e:
        print(f'Translation error: {e}')
        return [{'disease_en': '', 'treatment_en': '', 'effect_en': ''} for _ in rows]

print('✓ Translation functions ready.')

## Cell 4 — CSV Helpers

In [ ]:
OUTPUT_COLUMNS = [
    'entry_id',
    'arabic_name',
    'disease_or_condition',
    'disease_or_condition_en',
    'treatment_method',
    'treatment_method_en',
    'expected_effect',
    'expected_effect_en'
]

def load_output() -> pd.DataFrame:
    if os.path.exists(OUTPUT_CSV):
        return pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
    return pd.DataFrame(columns=OUTPUT_COLUMNS)

def save_batch(rows: list):
    df     = load_output()
    new_df = pd.DataFrame(rows, columns=OUTPUT_COLUMNS)
    df     = pd.concat([df, new_df], ignore_index=True)
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

def save_progress(idx: int):
    with open(PROGRESS_FILE, 'w') as f:
        f.write(str(idx))

def load_progress() -> int:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            return int(f.read().strip())
    return -1

print('✓ CSV helpers ready.')

## Cell 5 — Test on First 5 Rows
Always run this before the full loop.

In [ ]:
df_input = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
print(f'✓ Loaded {len(df_input)} rows')

# Test on first 5 rows
test_rows = []
for _, row in df_input.head(5).iterrows():
    test_rows.append({
        'disease':   str(row['disease_or_condition']),
        'treatment': str(row['treatment_method']),
        'effect':    str(row['expected_effect'])
    })

results = translate_batch(test_rows)

print('\nTest results:')
for i, (inp, out) in enumerate(zip(test_rows, results)):
    print(f'\nRow {i+1}:')
    print(f'  Disease   AR: {inp["disease"]}')
    print(f'  Disease   EN: {out["disease_en"]}')
    print(f'  Treatment AR: {inp["treatment"]}')
    print(f'  Treatment EN: {out["treatment_en"]}')
    print(f'  Effect    AR: {inp["effect"]}')
    print(f'  Effect    EN: {out["effect_en"]}')

## Cell 6 — Main Translation Loop

Translates all 7,362 rows in batches of 20.
Saves every 50 rows.
Handles placeholders without API calls.
Fully resumable.

In [ ]:
last_done  = load_progress()
resume_idx = last_done + 1
total      = len(df_input)
already_done = len(load_output())

print(f'Total rows        : {total}')
print(f'Already done      : {already_done}')
print(f'Remaining         : {total - resume_idx}')
print(f'Batch size        : {BATCH_SIZE}')
print('=' * 60)

save_buffer     = []   # Rows waiting to be saved
rows_translated = already_done

# Process in batches
idx = resume_idx
while idx < total:

    # Get next batch
    batch_end  = min(idx + BATCH_SIZE, total)
    batch_df   = df_input.iloc[idx:batch_end]

    # Separate placeholders from real translations
    api_rows   = []  # Rows that need Gemini
    api_indices = [] # Their positions in the batch

    batch_results = [None] * len(batch_df)

    for i, (_, row) in enumerate(batch_df.iterrows()):
        disease   = str(row['disease_or_condition'])
        treatment = str(row['treatment_method'])
        effect    = str(row['expected_effect'])

        # Check if all three are placeholders
        d_ph = translate_placeholder(disease)
        t_ph = translate_placeholder(treatment)
        e_ph = translate_placeholder(effect)

        # If disease needs translation send to API
        # Even if treatment/effect are placeholders
        if d_ph is None:  # disease needs real translation
            api_rows.append({
                'disease':   disease,
                'treatment': treatment if t_ph is None else '',
                'effect':    effect if e_ph is None else ''
            })
            api_indices.append(i)
        else:
            # All are placeholders or empty
            batch_results[i] = {
                'disease_en':   d_ph if d_ph is not None else '',
                'treatment_en': t_ph if t_ph is not None else '',
                'effect_en':    e_ph if e_ph is not None else ''
            }

    # Translate API rows
    if api_rows:
        api_results = translate_batch(api_rows)
        for i, result in zip(api_indices, api_results):
            # Fill in placeholder translations for treatment/effect if needed
            row = batch_df.iloc[i]
            t_ph = translate_placeholder(str(row['treatment_method']))
            e_ph = translate_placeholder(str(row['expected_effect']))

            batch_results[i] = {
                'disease_en':   result['disease_en'],
                'treatment_en': result['treatment_en'] if t_ph is None else t_ph,
                'effect_en':    result['effect_en'] if e_ph is None else e_ph
            }

    # Build output rows
    for i, (_, row) in enumerate(batch_df.iterrows()):
        result = batch_results[i] or {'disease_en': '', 'treatment_en': '', 'effect_en': ''}
        save_buffer.append({
            'entry_id':              row['entry_id'],
            'arabic_name':           row['arabic_name'],
            'disease_or_condition':  row['disease_or_condition'],
            'disease_or_condition_en': result['disease_en'],
            'treatment_method':      row['treatment_method'],
            'treatment_method_en':   result['treatment_en'],
            'expected_effect':       row['expected_effect'],
            'expected_effect_en':    result['effect_en']
        })

    rows_translated += len(batch_df)
    idx = batch_end

    # Save every 50 rows
    if len(save_buffer) >= 50 or idx >= total:
        save_batch(save_buffer)
        save_progress(idx - 1)
        save_buffer = []

        pct = round(rows_translated / total * 100, 1)
        print(
            f'[{rows_translated:>5}/{total}] {pct:>5}% | '
            f'Saved to Drive ✓'
        )

    time.sleep(0.5)

print('\n' + '=' * 60)
print('TRANSLATION COMPLETE')
print(f'Total rows translated : {rows_translated}')
print(f'Output                : {OUTPUT_CSV}')

## Cell 7 — Final Summary & Excel Export

In [ ]:
df_out = load_output()

print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'Total rows            : {len(df_out)}')
print(f'Unique herbs          : {df_out["entry_id"].nunique()}')
print(f'Empty disease_en      : {(df_out["disease_or_condition_en"] == "").sum()}')
print(f'Empty treatment_en    : {(df_out["treatment_method_en"] == "").sum()}')
print(f'Empty effect_en       : {(df_out["expected_effect_en"] == "").sum()}')
print()

# Sample check
print('Sample of 5 rows:')
for _, row in df_out.sample(5).iterrows():
    print(f"\n[{row['arabic_name']}]")
    print(f"  Disease AR : {row['disease_or_condition']}")
    print(f"  Disease EN : {row['disease_or_condition_en']}")
    print(f"  Treatment EN: {row['treatment_method_en']}")
    print(f"  Effect EN  : {row['expected_effect_en']}")

# Export Excel
EXCEL_PATH = f'{DRIVE_FOLDER}/disease_treatment_english_v1.xlsx'

with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:

    # Sheet 1 — Full bilingual (Arabic + English side by side)
    df_out.to_excel(
        writer, sheet_name='Bilingual_Full', index=False
    )

    # Sheet 2 — English only
    df_english = df_out[[
        'entry_id', 'arabic_name',
        'disease_or_condition_en',
        'treatment_method_en',
        'expected_effect_en'
    ]].copy()
    df_english.columns = [
        'entry_id', 'arabic_name',
        'disease_or_condition',
        'treatment_method',
        'expected_effect'
    ]
    df_english.to_excel(
        writer, sheet_name='English_Only', index=False
    )

    # Sheet 3 — Arabic only
    df_arabic = df_out[[
        'entry_id', 'arabic_name',
        'disease_or_condition',
        'treatment_method',
        'expected_effect'
    ]]
    df_arabic.to_excel(
        writer, sheet_name='Arabic_Only', index=False
    )

print(f'\n✓ Excel saved: {EXCEL_PATH}')
print(f'  Sheet 1 Bilingual_Full : {len(df_out)} rows')
print(f'  Sheet 2 English_Only   : {len(df_out)} rows')
print(f'  Sheet 3 Arabic_Only    : {len(df_out)} rows')